In [18]:
import pandas as pd

input_path = r"C:\Users\ariel\Desktop\Seminar - Thesis\F-TM-CR\data\accepted_2007_to_2018Q4.csv"

df_zip = pd.read_csv(
    input_path,
    usecols=["zip_code"],
    dtype={"zip_code": "string"},
    low_memory=False
)

# zip_code בלנדינגקלאב לרוב נראה כמו "123xx" -> ZIP3 = "123"
df_zip["zip3"] = df_zip["zip_code"].str.extract(r"(\d{3})", expand=False)
df_zip = df_zip.dropna(subset=["zip3"])
df_zip["zip3"] = df_zip["zip3"].str.zfill(3)

zip3_counts = df_zip["zip3"].value_counts().reset_index()
zip3_counts.columns = ["zip3", "count"]

zip3_counts.to_csv("zip3_counts_from_lc.csv", index=False)
print(f"Saved zip3_counts_from_lc.csv | unique ZIP3: {len(zip3_counts)}")


Saved zip3_counts_from_lc.csv | unique ZIP3: 956


In [19]:
import os
import time
import requests
import pandas as pd

API_KEY = os.getenv("CENSUS_API_KEY")
if not API_KEY:
    raise RuntimeError("CENSUS_API_KEY not found in environment variables.")

YEARS = [2011, 2012, 2013, 2014]
OUT_DIR = r".\census_final_data"
os.makedirs(OUT_DIR, exist_ok=True)

# משתנים:
# B01003_001E = total population
# B19013_001E = median household income
# B03002_004E = Not Hispanic or Latino: Black alone
# B03002_012E = Hispanic or Latino
# B17001_002E = Below poverty level
# B17001_001E = Poverty universe (Total for whom poverty status is determined)
# B11001_001E = Total households
VAR_CODES = {
    "B01003_001E": "total_pop",
    "B19013_001E": "median_income",
    "B03002_004E": "black_nh",
    "B03002_012E": "hispanic",
    "B17001_002E": "poverty_count",
    "B17001_001E": "poverty_universe",
    "B11001_001E": "total_households",
}

def fetch_year_all_vars(year: int) -> pd.DataFrame:
    base_url = f"https://api.census.gov/data/{year}/acs/acs5"
    get_list = ",".join(["NAME"] + list(VAR_CODES.keys()))
    params = {
        "get": get_list,
        "for": "zip code tabulation area:*",
        "key": API_KEY
    }

    r = requests.get(base_url, params=params, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Census API error {r.status_code}: {r.text[:300]}")

    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])

    # rename
    df = df.rename(columns={"zip code tabulation area": "zcta", **VAR_CODES})


    # numeric clean
    for c in VAR_CODES.values():
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df.loc[df[c] < 0, c] = pd.NA   # <-- זה התיקון: קודים שליליים = NA


    # ZCTA -> ZIP3
    df["zcta"] = df["zcta"].astype(str).str.zfill(5)
    df["zip3"] = df["zcta"].str[:3]

    df["year"] = year
    return df[["year", "zcta", "zip3"] + list(VAR_CODES.values())]

all_years = []
for year in YEARS:
    print(f"Fetching {year} ...")
    df_year = fetch_year_all_vars(year)
    all_years.append(df_year)
    time.sleep(0.4)

raw = pd.concat(all_years, ignore_index=True)

# אגרגציה ל-ZIP3
# חשוב: median_income הוא מדיאן ולכן לא באמת "מתחבר".
# אנחנו עושים קירוב: ממוצע משוקלל לפי משקי בית (total_households).
def build_zip3_panel(df: pd.DataFrame) -> pd.DataFrame:
    # סכומים
    sums = (
        df.groupby(["year", "zip3"], as_index=False)[
            ["total_pop", "black_nh", "hispanic", "poverty_count", "poverty_universe", "total_households"]
        ].sum(min_count=1)
    )

    # weighted "median income approx"
    inc = df.dropna(subset=["median_income", "total_households"]).copy()
    inc = inc[(inc["total_households"] > 0) & (inc["median_income"] > 0)]

    w_inc = (
        inc.groupby(["year", "zip3"], as_index=False)
           .apply(lambda x: (x["median_income"] * x["total_households"]).sum() / x["total_households"].sum())
           .rename(columns={None: "median_income_approx"})
    )

    out = sums.merge(w_inc, on=["year", "zip3"], how="left")

    # shares / rates
    out["share_black"] = out["black_nh"] / out["total_pop"]
    out["share_hispanic"] = out["hispanic"] / out["total_pop"]

    # תיקון קריטי: poverty_rate לפי universe של עוני
    out["poverty_rate"] = out["poverty_count"] / out["poverty_universe"]
    out.loc[out["poverty_universe"].isna() | (out["poverty_universe"] <= 0), "poverty_rate"] = pd.NA

    return out

zip3_panel = build_zip3_panel(raw)

zip3_panel_path = os.path.join(OUT_DIR, "zip3_census_panel_2011_2014.csv")
zip3_panel.to_csv(zip3_panel_path, index=False)
print(f"Saved: {zip3_panel_path} | rows: {len(zip3_panel)}")


Fetching 2011 ...


ReadTimeout: HTTPSConnectionPool(host='api.census.gov', port=443): Read timed out. (read timeout=60)

In [6]:
import os
import pandas as pd

OUT_DIR = r".\census_final_data"
panel = pd.read_csv(os.path.join(OUT_DIR, "zip3_census_panel_2011_2014.csv"), dtype={"zip3":"string"})
panel["zip3"] = panel["zip3"].str.zfill(3)

lc_zip3 = pd.read_csv("zip3_counts_from_lc.csv", dtype={"zip3":"string"})
lc_zip3["zip3"] = lc_zip3["zip3"].str.zfill(3)

panel_zip3_set = set(panel["zip3"].unique())
lc_zip3_set = set(lc_zip3["zip3"].unique())

matched = lc_zip3_set.intersection(panel_zip3_set)
match_rate = len(matched) / len(lc_zip3_set) if lc_zip3_set else 0

report = (
    f"LC unique ZIP3: {len(lc_zip3_set)}\n"
    f"Panel unique ZIP3: {len(panel_zip3_set)}\n"
    f"Matched ZIP3: {len(matched)}\n"
    f"Match rate: {match_rate:.3%}\n"
)

with open(os.path.join(OUT_DIR, "coverage_report.txt"), "w", encoding="utf-8") as f:
    f.write(report)

print(report)

# פרופיל ZIP3 ממוצע על השנים (למיזוג עתידי כ-feature)
zip3_profile = (
    panel.groupby("zip3", as_index=False)[
        ["share_black","share_hispanic","poverty_rate","median_income_approx","total_pop"]
    ].mean()
)

zip3_profile.to_csv(os.path.join(OUT_DIR, "zip3_census_profile_mean.csv"), index=False)
print("Saved: zip3_census_profile_mean.csv")


LC unique ZIP3: 956
Panel unique ZIP3: 894
Matched ZIP3: 890
Match rate: 93.096%

Saved: zip3_census_profile_mean.csv


In [13]:
import pandas as pd
import numpy as np

prof = pd.read_csv(r".\census_final_data\zip3_census_profile_mean.csv", dtype={"zip3":"string"})
prof["income_decile"]  = pd.qcut(prof["median_income_approx"], 10, labels=False, duplicates="drop") + 1
prof["poverty_decile"] = pd.qcut(prof["poverty_rate"], 10, labels=False, duplicates="drop") + 1
prof["high_black"]     = (prof["share_black"] >= prof["share_black"].quantile(0.8)).astype(int)

prof.to_csv(r".\census_final_data\zip3_groups.csv", index=False)
print("Saved: zip3_groups.csv")


Saved: zip3_groups.csv


In [14]:
import pandas as pd
import numpy as np

panel = pd.read_csv(r".\census_final_data\zip3_census_panel_2011_2014.csv", dtype={"zip3":"string"})
panel["zip3"] = panel["zip3"].str.zfill(3)

# sanity checks
print("Rows:", len(panel))
print("ZIP3 unique:", panel["zip3"].nunique())

# poverty_rate should be between 0 and 1, and poverty_universe should be >0
bad_universe = (panel["poverty_universe"].isna() | (panel["poverty_universe"] <= 0)).sum()
bad_rate = ((panel["poverty_rate"] < 0) | (panel["poverty_rate"] > 1) | panel["poverty_rate"].isna()).sum()

print("Bad poverty_universe (<=0/NA):", bad_universe)
print("Bad poverty_rate (<0/>1/NA):", bad_rate)

print(panel[["poverty_rate","median_income_approx","share_black","share_hispanic"]].describe())


Rows: 3576
ZIP3 unique: 894
Bad poverty_universe (<=0/NA): 20
Bad poverty_rate (<0/>1/NA): 20
       poverty_rate  median_income_approx  share_black  share_hispanic
count   3556.000000           3556.000000  3556.000000     3556.000000
mean       0.156189          52503.040929     0.095875        0.116785
std        0.057603          15265.160798     0.123999        0.147976
min        0.034167          15635.996907     0.000298        0.001362
25%        0.117384          43020.769989     0.013991        0.026916
50%        0.152513          49125.823340     0.038697        0.057505
75%        0.186699          58135.542291     0.129349        0.144457
max        0.524198         181316.957105     0.704163        0.994867


In [ ]:
import os
import pandas as pd

OUT_DIR = r".\census_final_data"

panel = pd.read_csv(os.path.join(OUT_DIR, "zip3_census_panel_2011_2014.csv"), dtype={"zip3":"string"})
panel["zip3"] = panel["zip3"].str.zfill(3)

lc_zip3 = pd.read_csv("zip3_counts_from_lc.csv", dtype={"zip3":"string"})
lc_zip3["zip3"] = lc_zip3["zip3"].str.zfill(3)

panel_zip3_set = set(panel["zip3"].dropna().unique())
lc_zip3_set = set(lc_zip3["zip3"].dropna().unique())

matched = lc_zip3_set & panel_zip3_set
missing_in_panel = sorted(lc_zip3_set - panel_zip3_set)

match_rate = len(matched) / len(lc_zip3_set) if lc_zip3_set else 0

report = (
    f"LC unique ZIP3: {len(lc_zip3_set)}\n"
    f"Panel unique ZIP3: {len(panel_zip3_set)}\n"
    f"Matched ZIP3: {len(matched)}\n"
    f"Missing in panel: {len(missing_in_panel)}\n"
    f"Match rate: {match_rate:.3%}\n"
)

with open(os.path.join(OUT_DIR, "coverage_report.txt"), "w", encoding="utf-8") as f:
    f.write(report)

pd.DataFrame({"zip3": missing_in_panel}).to_csv(os.path.join(OUT_DIR, "missing_zip3_in_panel.csv"), index=False)

print(report)
print("Saved: missing_zip3_in_panel.csv")


LC unique ZIP3: 956
Panel unique ZIP3: 894
Matched ZIP3: 890
Missing in panel: 66
Match rate: 93.096%

Saved: missing_zip3_in_panel.csv
